In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os.path
from pathlib import Path
import pickle
import multiprocessing
import time
import gc
from tqdm import tqdm

In [2]:
import import_ipynb

In [3]:
import DTW

In [4]:
import NWTW

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [5]:
import FlexDTW

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [6]:
DATASET = 'train' # 'test'
VERSION = 'toy'

In [7]:
QUERY_LIST = Path(f'cfg_files/queries.{DATASET}.{VERSION}')

In [8]:
SYSTEMS = ['dtw1', 'dtw2', 'dtw3', 'subseqdtw1', 'subseqdtw2', 'subseqdtw3', 'nwtw', 'flexdtw']
BENCHMARKS = ['matching', 'subseq_20', 'subseq_30', 'subseq_40', 'partialStart', 'partialEnd', 'partialOverlap', 
              'pre_5', 'pre_10', 'pre_20', 'post_5', 'post_10', 'post_20', 'prepost_5', 'prepost_10',
              'prepost_20']

In [9]:
features_root = Path('../ttmp/Chopin_Mazurkas_features')
FEAT_DIRS = {}

for benchmark in BENCHMARKS:
    if benchmark == 'partialOverlap':
        FEAT_DIRS[benchmark] = ([features_root/'partialStart', features_root/'partialEnd'])
    elif 'prepost' in benchmark:
        sec = benchmark.split('_')[-1]
        FEAT_DIRS[benchmark] = ([features_root/f'pre_{sec}', features_root/f'post_{sec}'])
    else:
        FEAT_DIRS[benchmark] = [features_root/f'{benchmark}', features_root/'original']

In [10]:
steps = {'dtw1': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'dtw2': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'dtw3': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'subseqdtw1': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'subseqdtw2': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'subseqdtw3': np.array([1,1,1,2,2,1]).reshape((-1,2)),
        'nwtw': 0, # transitions are specified in NWTW algorithm
        'flexdtw': np.array([1,1,1,2,2,1]).reshape((-1,2))
        }
weights = {'dtw1': np.array([2,3,3]),
          'dtw2': np.array([1,1,1]),
          'dtw3': np.array([1,2,2]),
          'subseqdtw1': np.array([1,1,2]),
          'subseqdtw2': np.array([2,3,3]),
          'subseqdtw3': np.array([1,2,2]),
          'nwtw': 0, # weights are specified in NWTW algorithm
          'flexdtw': np.array([1.25,3,3])
          }
other_params = {
                'flexdtw': {'beta': 0.1}
               }

# Benchmarks

In [11]:
def get_outfile(outdir, benchmark, system, queryid):
    outpath = (outdir / benchmark / system)
    outpath.mkdir(parents=True, exist_ok=True)
    outfile = (outpath / queryid).with_suffix('.pkl')
    return outfile

In [12]:
import numpy as np

def convert_edge_to_local(edge_type, position, chunk_shape):
    """
    Convert edge representation to local chunk coordinates.
    """
    if edge_type == 0:  # Top edge
        return chunk_shape[0] - 1, position  # Last row of actual chunk
    else:  # Right edge (edge_type == 1)
        return position, chunk_shape[1] - 1  # Last column of actual chunk


def convert_local_to_global(chunk_i, chunk_j, local_row, local_col, chunks_dict):
    """
    Convert local chunk coordinates to global matrix coordinates.
    
    Parameters:
    -----------
    chunk_i, chunk_j : int, int
        Chunk indices
    local_row, local_col : int, int
        Local coordinates within the chunk
    chunks_dict : dict
        Dictionary containing chunk data with 'hop' information
        
    Returns:
    --------
    global_row, global_col : int, int
        Global matrix coordinates
    """
    # Get the starting bounds for this chunk
    start_1, _, start_2, _ = chunks_dict[(chunk_i, chunk_j)]['bounds']
    
    global_row = start_1 + local_row
    global_col = start_2 + local_col
    return global_row, global_col


def get_starting_point(S_single, end_row, end_col):
    """
    Get the starting point for a path ending at (end_row, end_col),
    based on the 2D signed encoding from FlexDTW.

    If S_single[r, c] > 0 → start on bottom edge: (0, S_single[r, c])
    If S_single[r, c] < 0 → start on left edge: (-S_single[r, c], 0)
    """
    val = S_single[end_row, end_col]
    
    if val > 0:
        return 0, int(val)        # start on bottom edge
    elif val < 0:
        return abs(int(val)), 0    # start on left edge
    else:
        return 0, 0  # default / fallback (top-left corner)


def is_on_bottom_edge(start_row, start_col, chunk_shape):
    """Check if starting point is on bottom edge."""
    return start_row == 0


def is_on_left_edge(start_row, start_col, chunk_shape):
    """Check if starting point is on left edge."""
    return start_col == 0


def global_to_prev_chunk_edge(global_row, global_col, prev_chunk_i, prev_chunk_j, 
                               chunks_dict, L):
    """
    Convert global coordinates to edge position in previous chunk.
    
    Parameters:
    -----------
    global_row, global_col : int, int
        Global coordinates
    prev_chunk_i, prev_chunk_j : int, int
        Previous chunk indices
    chunks_dict : dict
        Dictionary containing chunk data
    L : int
        Chunk size
        
    Returns:
    --------
    edge_type, position : int, int
        Edge representation in previous chunk
    """
    # Get the bounds of the previous chunk
    start_1, _, start_2, _ = chunks_dict[(prev_chunk_i, prev_chunk_j)]['bounds']
    
    # Convert to local coordinates in previous chunk
    local_row = global_row - start_1
    local_col = global_col - start_2
    
    # Determine which edge (top or right) of previous chunk
    prev_chunk_shape = chunks_dict[(prev_chunk_i, prev_chunk_j)]['D'].shape
    
    if local_row == prev_chunk_shape[0] - 1:  # On top edge of previous chunk
        return 0, local_col
    elif local_col == prev_chunk_shape[1] - 1:  # On right edge of previous chunk
        return 1, local_row
    else:
        raise ValueError(f"Coordinates ({local_row}, {local_col}) not on edge of previous chunk")

def initialize_chunks(chunks_dict, num_chunks_1, num_chunks_2, L):
    """
    Initialize D_chunks and L_chunks for the first row and first column.
    Uses flexible data structure to accommodate non-square boundary chunks.
    
    Parameters:
    -----------
    chunks_dict : dict
        Dictionary containing chunk data (including 'hop' for each chunk)
    num_chunks_1, num_chunks_2 : int, int
        Number of chunks in each dimension
    L : int
        Chunk size
        
    Returns:
    --------
    D_chunks, L_chunks : list of list of dicts of lists
        Indexed as [chunk_row][chunk_col][edge_type][position]
        Where position is a Python list that can vary in length
    """
    # Initialize as nested lists with dicts for edge types
    D_chunks = [[{0: [], 1: []} for _ in range(num_chunks_2)] for _ in range(num_chunks_1)]
    L_chunks = [[{0: [], 1: []} for _ in range(num_chunks_2)] for _ in range(num_chunks_1)]
    
    # Helper function to get actual edge length for a chunk
    def get_edge_length(chunk_shape, edge_type):
        """Returns the actual length of an edge for a given chunk shape."""
        if edge_type == 0:  # top edge
            return chunk_shape[1]  # width
        else:  # right edge (edge_type == 1)
            return chunk_shape[0]  # height
    
    # Initialize chunk (0, 0)
    D_single_00 = chunks_dict[(0, 0)]['D']
    S_single_00 = chunks_dict[(0, 0)]['S']
    
    for edge_type in range(2):
        edge_len = get_edge_length(D_single_00.shape, edge_type)
        # Initialize lists with inf values
        D_chunks[0][0][edge_type] = [np.inf] * edge_len
        L_chunks[0][0][edge_type] = [np.inf] * edge_len
        
        for position in range(edge_len):
            local_row, local_col = convert_edge_to_local(edge_type, position, D_single_00.shape)
            
            if local_row < D_single_00.shape[0] and local_col < D_single_00.shape[1]:
                start_row, start_col = get_starting_point(S_single_00, local_row, local_col)
                
                if is_on_bottom_edge(start_row, start_col, D_single_00.shape) or \
                   is_on_left_edge(start_row, start_col, D_single_00.shape):
                    D_chunks[0][0][edge_type][position] = D_single_00[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[0][0][edge_type][position] = path_length
    
    # Initialize first row: chunks (0, j) for j = 1, 2, ...
    for j in range(1, num_chunks_2):
        if (0, j) not in chunks_dict:
            continue
            
        D_single = chunks_dict[(0, j)]['D']
        S_single = chunks_dict[(0, j)]['S'] 
        C_chunk = chunks_dict[(0, j)]['C'] if 'C' in chunks_dict[(0, j)] else None
        
        for edge_type in range(2):
            edge_len = get_edge_length(D_single.shape, edge_type)
            D_chunks[0][j][edge_type] = [np.inf] * edge_len
            L_chunks[0][j][edge_type] = [np.inf] * edge_len
            
            for position in range(edge_len):
                local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                
                if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                    continue
                
                start_row, start_col = get_starting_point(S_single, local_row, local_col)
                
                # Case 1: Valid start on bottom edge
                if is_on_bottom_edge(start_row, start_col, D_single.shape):
                    D_chunks[0][j][edge_type][position] = D_single[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[0][j][edge_type][position] = path_length
                
                # Case 2: Start on left edge - need cost from previous chunk
                elif is_on_left_edge(start_row, start_col, D_single.shape):
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        0, j, start_row, start_col, chunks_dict
                    )
                    
                    # Map to previous chunk (0, j-1)
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, 0, j - 1, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[0][j-1][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get cost from previous chunk
                    prev_cost = D_chunks[0][j-1][prev_edge_type][prev_position]
                    
                    if np.isfinite(prev_cost):  # Only if valid path exists
                        # Subtract overlapping cell cost from CURRENT chunk to avoid double counting
                        overlap_cost = C_chunk[start_row, 0] if C_chunk is not None else 0
                        
                        D_chunks[0][j][edge_type][position] = D_single[local_row, local_col] + prev_cost - overlap_cost 
                        # Path length includes previous chunk
                        prev_length = L_chunks[0][j-1][prev_edge_type][prev_position]
                        curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[0][j][edge_type][position] = prev_length + curr_length
                        
                        if edge_type == 0 and position == 0 and j > 0:
                            D_chunks[0][j][0][0] = D_chunks[0][j-1][0][-1]
                            L_chunks[0][j][0][0] = L_chunks[0][j-1][0][-1]
    
    # Initialize first column: chunks (i, 0) for i = 1, 2, ...
    for i in range(1, num_chunks_1):
        if (i, 0) not in chunks_dict:
            continue
            
        D_single = chunks_dict[(i, 0)]['D']
        S_single = chunks_dict[(i, 0)]['S']
        C_chunk = chunks_dict[(i, 0)]['C'] if 'C' in chunks_dict[(i, 0)] else None
        
        for edge_type in range(2):
            edge_len = get_edge_length(D_single.shape, edge_type)
            D_chunks[i][0][edge_type] = [np.inf] * edge_len
            L_chunks[i][0][edge_type] = [np.inf] * edge_len
            
            for position in range(edge_len):
                local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                
                if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                    continue
                
                start_row, start_col = get_starting_point(S_single, local_row, local_col)
                
                # Case 1: Valid start on left edge
                if is_on_left_edge(start_row, start_col, D_single.shape):
                    D_chunks[i][0][edge_type][position] = D_single[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[i][0][edge_type][position] = path_length
                
                # Case 2: Start on bottom edge - need cost from previous chunk
                elif is_on_bottom_edge(start_row, start_col, D_single.shape):
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        i, 0, start_row, start_col, chunks_dict
                    )
                    
                    # Map to previous chunk (i-1, 0)
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, i - 1, 0, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[i-1][0][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get cost from previous chunk
                    prev_cost = D_chunks[i-1][0][prev_edge_type][prev_position]
                    
                    if np.isfinite(prev_cost):  # Only if valid path exists
                        # Subtract overlapping cell cost from CURRENT chunk
                        overlap_cost = C_chunk[0, start_col] if C_chunk is not None else 0
                        
                        D_chunks[i][0][edge_type][position] = D_single[local_row, local_col] + prev_cost - overlap_cost
                        
                        # Path length includes previous chunk
                        prev_length = L_chunks[i-1][0][prev_edge_type][prev_position]
                        curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[i][0][edge_type][position] = prev_length + curr_length
                        if i > 0 and edge_type == 1 and position == 0 :
                            D_chunks[i][0][1][0] = D_chunks[i-1][0][1][-1]
                            L_chunks[i][0][1][0] = L_chunks[i-1][0][1][-1]
    
    return D_chunks, L_chunks


def dp_fill_chunks(chunks_dict, D_chunks, L_chunks, num_chunks_1, num_chunks_2, L):
    """
    Fill in D_chunks and L_chunks for all interior chunks using dynamic programming.
    Uses flexible hop sizes from chunks_dict.
    
    Parameters:
    -----------
    chunks_dict : dict
        Dictionary containing chunk data (including flexible 'hop' per chunk)
    D_chunks, L_chunks : list of list of dicts of lists
        Chunk-level cost and length tensors (partially filled)
    num_chunks_1, num_chunks_2 : int, int
        Number of chunks in each dimension
    L : int
        Standard chunk size (for reference)
    """
    # plot_normalized_global_edge_cost(D_chunks, L_chunks, num_chunks_1, num_chunks_2)
    # Helper function to get actual edge length for a chunk
    def get_edge_length(chunk_shape, edge_type):
        """Returns the actual length of an edge for a given chunk shape."""
        if edge_type == 0:  # top edge
            return chunk_shape[1]  # width
        else:  # right edge (edge_type == 1)
            return chunk_shape[0]  # height
    
    # Process chunks row by row
    for i in range(num_chunks_1):
        for j in range(num_chunks_2):
            # Skip first row and first column (already initialized)
            if i == 0 or j == 0:
                continue
            
            D_single = chunks_dict[(i, j)]['D']
            S_single = chunks_dict[(i, j)]['S']
            C_chunk = chunks_dict[(i, j)]['C'] if 'C' in chunks_dict[(i, j)] else None
            
            for edge_type in range(2):
                edge_len = get_edge_length(D_single.shape, edge_type)
                # Initialize lists for this chunk if not already done
                D_chunks[i][j][edge_type] = [np.inf] * edge_len
                L_chunks[i][j][edge_type] = [np.inf] * edge_len
                
                for position in range(edge_len):
                    
                        
                        # plot_normalized_global_edge_cost(D_chunks, L_chunks, num_chunks_1, num_chunks_2)
                    local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                    
                    if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                        continue
                    
                    # Get starting point for this path
                    start_row, start_col = get_starting_point(S_single, local_row, local_col)
                    
                    # Determine which previous chunk this path came from
                    if is_on_bottom_edge(start_row, start_col, D_single.shape):
                        # Path came from chunk above (i-1, j)
                        prev_i, prev_j = i - 1, j
                    elif is_on_left_edge(start_row, start_col, D_single.shape):
                        # Path came from chunk to the left (i, j-1)
                        prev_i, prev_j = i, j - 1
                    else:
                        # Path started within this chunk
                        D_chunks[i][j][edge_type][position] = D_single[local_row, local_col]
                        path_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[i][j][edge_type][position] = path_length
                        continue
                    
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        i, j, start_row, start_col, chunks_dict
                    )
                    
                    # Map to edge position in previous chunk
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, prev_i, prev_j, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[prev_i][prev_j][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get accumulated cost and length from previous chunk
                    prev_cost = D_chunks[prev_i][prev_j][prev_edge_type][prev_position]
                    prev_length = L_chunks[prev_i][prev_j][prev_edge_type][prev_position]
                    
                    # Only proceed if valid path exists in previous chunk
                    if not np.isfinite(prev_cost):
                        continue
                    
                    # Current chunk contribution (subtract first cell to avoid double counting)
                    if C_chunk is not None:
                        first_cell_cost = C_chunk[start_row, start_col]
                    else:
                        first_cell_cost = 0
                    
                    curr_cost_contribution = D_single[local_row, local_col] - first_cell_cost
                    curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                    
                    # Propagate forward
                    
                    D_chunks[i][j][edge_type][position] = prev_cost + curr_cost_contribution
                    if position%4000==0:
                        print(D_chunks[i][j][edge_type][position])
                        print(i,j,edge_type,position)
                    L_chunks[i][j][edge_type][position] = prev_length + curr_length
                    
                    if j > 0 and edge_type == 0 & position == 0:
                        D_chunks[i][j][0][0] = D_chunks[i][j-1][0][-1]
                        L_chunks[i][j][0][0] = L_chunks[i][j-1][0][-1]
                    if i > 0 and edge_type == 1 & position == 0:
                        D_chunks[i][j][1][0] = D_chunks[i-1][j][1][-1]
                        L_chunks[i][j][1][0] = L_chunks[i-1][j][1][-1]
                 
    
    return D_chunks, L_chunks

def chunk_flexdtw(C, L, steps=None, weights=None, buffer=1):
    """
    Break cost matrix C into overlapping chunks and run FlexDTW on each.
    Ensures exactly 1-cell overlap between chunks. Last chunks may be smaller
    than L×L to maintain the 1-cell overlap constraint.
    
    Now stores flexible hop information for each chunk to handle boundary truncation.
    
    Parameters:
    -----------
    C : ndarray
        Cost matrix of shape (L1, L2)
    L : int
        Standard chunk size
    steps : list, optional
        Step patterns for DTW
    weights : list, optional
        Weights for each step pattern
    buffer : int
        Buffer parameter for FlexDTW
        
    Returns:
    --------
    chunks_dict : dict
        Dictionary with chunk data, including 'bounds' and 'hop' for each chunk
    L : int
        Chunk size
    n_chunks_1, n_chunks_2 : int
        Number of chunks in each dimension
    """
    import math
    
    if steps is None:
        steps = [(1,1), (1,2), (2,1)]
    if weights is None:
        weights = [2, 3, 3]
    
    L1, L2 = C.shape
    print(f"Matrix shape: {L1} × {L2}")
    hop = L - 1  # Standard overlap of 1
    
    # Compute number of chunks along each axis
    n_chunks_1 = math.ceil((L1 - 1) / hop)
    n_chunks_2 = math.ceil((L2 - 1) / hop)
    
    chunks_dict = {}
    
    for i in range(n_chunks_1):
        for j in range(n_chunks_2):
            # Standard start positions with hop size
            start_1 = i * hop
            start_2 = j * hop
            
            # Standard end positions
            end_1 = start_1 + L
            end_2 = start_2 + L
            
            # For boundary chunks, don't shift start - just truncate end
            # This ensures we always have exactly 1-frame overlap
            if end_1 > L1:
                end_1 = L1
            
            if end_2 > L2:
                end_2 = L2
            
            # Extract chunk
            C_chunk = C[int(start_1):int(end_1), int(start_2):int(end_2)]
            
            # Run FlexDTW on this chunk
            # Note: You'll need to import or have FlexDTW available
            # Assuming FlexDTW.flexdtw returns (best_cost, wp, D, P, B, debug)
            try:
                import FlexDTW
                best_cost, wp, D, P, B, debug = FlexDTW.flexdtw(
                    C_chunk, 
                    steps=steps, 
                    weights=weights, 
                    buffer=buffer
                )
            except ImportError:
                # Placeholder if FlexDTW not available
                print("Warning: FlexDTW not imported, using placeholder values")
                best_cost = 0
                wp = []
                D = np.zeros_like(C_chunk)
                P = np.zeros_like(C_chunk)
                B = np.zeros_like(C_chunk)
                debug = {}
            
            # Calculate actual hop for this chunk
            # For boundary chunks, the hop to the next chunk may be different
            actual_hop_1 = hop if end_1 < L1 else (L1 - start_1)
            actual_hop_2 = hop if end_2 < L2 else (L2 - start_2)
            
            chunks_dict[(i, j)] = {
                'C': C_chunk,
                'D': D,
                'S': P,  # Start positions (signed encoding)
                'B': B,  # Backpointer matrix
                'debug': debug,
                'best_cost': best_cost,
                'wp': wp,
                'bounds': (start_1, end_1, start_2, end_2),
                'hop': (actual_hop_1, actual_hop_2),  # Store flexible hop sizes
                'shape': C_chunk.shape
            }
            
            print(f"Chunk ({i},{j}): [{start_1}:{end_1}, {start_2}:{end_2}], "
                  f"shape={C_chunk.shape}, hop=({actual_hop_1},{actual_hop_2}), "
                  f"best_cost={best_cost:.4f}")
   
    return chunks_dict, L, n_chunks_1, n_chunks_2

In [13]:
def align_system(system, F1, F2, outfile, L):
    
    subseq = 'subseq' in system
    
    if system == 'flexdtw':
        L1 = F1.shape[1]
        L2 = F2.shape[1]
        buffer_global = min(L1, L2) * (1 - (1 - other_params[system]['beta']) * min(L1,L2) / max(L1, L2))
        C = 1 - FlexDTW.L2norm(F1).T @ FlexDTW.L2norm(F2)
        L1, L2 = C.shape
         
        # Define chunk size L
        L = L
        
        # Run chunked FlexDTW
        chunks_dict, L, n_chunks_1, n_chunks_2 = chunk_flexdtw(
            C, 
            L=L, 
            steps=[(1,1), (1,2), (2,1)], 
            weights=[1.5, 3.0, 3.0], 
            buffer=1
        ) 
            
    
    return chunks_dict,  L, n_chunks_1, n_chunks_2

In [14]:
import numpy as np

def convert_edge_to_local(edge_type, position, chunk_shape):
    """
    Convert edge representation to local chunk coordinates.
    """
    if edge_type == 0:  # Top edge
        return chunk_shape[0] - 1, position  # Last row of actual chunk
    else:  # Right edge (edge_type == 1)
        return position, chunk_shape[1] - 1  # Last column of actual chunk


def convert_local_to_global(chunk_i, chunk_j, local_row, local_col, chunks_dict):
    """
    Convert local chunk coordinates to global matrix coordinates.
    
    Parameters:
    -----------
    chunk_i, chunk_j : int, int
        Chunk indices
    local_row, local_col : int, int
        Local coordinates within the chunk
    chunks_dict : dict
        Dictionary containing chunk data with 'hop' information
        
    Returns:
    --------
    global_row, global_col : int, int
        Global matrix coordinates
    """
    # Get the starting bounds for this chunk
    start_1, _, start_2, _ = chunks_dict[(chunk_i, chunk_j)]['bounds']
    
    global_row = start_1 + local_row
    global_col = start_2 + local_col
    return global_row, global_col


def get_starting_point(S_single, end_row, end_col):
    """
    Get the starting point for a path ending at (end_row, end_col),
    based on the 2D signed encoding from FlexDTW.

    If S_single[r, c] > 0 → start on bottom edge: (0, S_single[r, c])
    If S_single[r, c] < 0 → start on left edge: (-S_single[r, c], 0)
    """
    val = S_single[end_row, end_col]
    
    if val > 0:
        return 0, int(val)        # start on bottom edge
    elif val < 0:
        return abs(int(val)), 0    # start on left edge
    else:
        return 0, 0  # default / fallback (top-left corner)


def is_on_bottom_edge(start_row, start_col, chunk_shape):
    """Check if starting point is on bottom edge."""
    return start_row == 0


def is_on_left_edge(start_row, start_col, chunk_shape):
    """Check if starting point is on left edge."""
    return start_col == 0


def global_to_prev_chunk_edge(global_row, global_col, prev_chunk_i, prev_chunk_j, 
                               chunks_dict, L):
    """
    Convert global coordinates to edge position in previous chunk.
    
    Parameters:
    -----------
    global_row, global_col : int, int
        Global coordinates
    prev_chunk_i, prev_chunk_j : int, int
        Previous chunk indices
    chunks_dict : dict
        Dictionary containing chunk data
    L : int
        Chunk size
        
    Returns:
    --------
    edge_type, position : int, int
        Edge representation in previous chunk
    """
    # Get the bounds of the previous chunk
    start_1, _, start_2, _ = chunks_dict[(prev_chunk_i, prev_chunk_j)]['bounds']
    
    # Convert to local coordinates in previous chunk
    local_row = global_row - start_1
    local_col = global_col - start_2
    
    # Determine which edge (top or right) of previous chunk
    prev_chunk_shape = chunks_dict[(prev_chunk_i, prev_chunk_j)]['D'].shape
    
    if local_row == prev_chunk_shape[0] - 1:  # On top edge of previous chunk
        return 0, local_col
    elif local_col == prev_chunk_shape[1] - 1:  # On right edge of previous chunk
        return 1, local_row
    else:
        raise ValueError(f"Coordinates ({local_row}, {local_col}) not on edge of previous chunk")


def initialize_chunks(chunks_dict, num_chunks_1, num_chunks_2, L):
    """
    Initialize D_chunks and L_chunks for the first row and first column.
    Uses flexible data structure to accommodate non-square boundary chunks.
    
    Parameters:
    -----------
    chunks_dict : dict
        Dictionary containing chunk data (including 'hop' for each chunk)
    num_chunks_1, num_chunks_2 : int, int
        Number of chunks in each dimension
    L : int
        Chunk size
        
    Returns:
    --------
    D_chunks, L_chunks : list of list of dicts of lists
        Indexed as [chunk_row][chunk_col][edge_type][position]
        Where position is a Python list that can vary in length
    """
    # Initialize as nested lists with dicts for edge types
    D_chunks = [[{0: [], 1: []} for _ in range(num_chunks_2)] for _ in range(num_chunks_1)]
    L_chunks = [[{0: [], 1: []} for _ in range(num_chunks_2)] for _ in range(num_chunks_1)]
    
    # Helper function to get actual edge length for a chunk
    def get_edge_length(chunk_shape, edge_type):
        """Returns the actual length of an edge for a given chunk shape."""
        if edge_type == 0:  # top edge
            return chunk_shape[1]  # width
        else:  # right edge (edge_type == 1)
            return chunk_shape[0]  # height
    
    # Initialize chunk (0, 0)
    D_single_00 = chunks_dict[(0, 0)]['D']
    S_single_00 = chunks_dict[(0, 0)]['S']
    
    for edge_type in range(2):
        edge_len = get_edge_length(D_single_00.shape, edge_type)
        # Initialize lists with inf values
        D_chunks[0][0][edge_type] = [np.inf] * edge_len
        L_chunks[0][0][edge_type] = [np.inf] * edge_len
        
        for position in range(edge_len):
            local_row, local_col = convert_edge_to_local(edge_type, position, D_single_00.shape)
            
            if local_row < D_single_00.shape[0] and local_col < D_single_00.shape[1]:
                start_row, start_col = get_starting_point(S_single_00, local_row, local_col)
                
                if is_on_bottom_edge(start_row, start_col, D_single_00.shape) or \
                   is_on_left_edge(start_row, start_col, D_single_00.shape):
                    D_chunks[0][0][edge_type][position] = D_single_00[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[0][0][edge_type][position] = path_length
    
    # Initialize first row: chunks (0, j) for j = 1, 2, ...
    for j in range(1, num_chunks_2):
        if (0, j) not in chunks_dict:
            continue
            
        D_single = chunks_dict[(0, j)]['D']
        S_single = chunks_dict[(0, j)]['S'] 
        C_chunk = chunks_dict[(0, j)]['C'] if 'C' in chunks_dict[(0, j)] else None
        
        for edge_type in range(2):
            edge_len = get_edge_length(D_single.shape, edge_type)
            D_chunks[0][j][edge_type] = [np.inf] * edge_len
            L_chunks[0][j][edge_type] = [np.inf] * edge_len
            
            for position in range(edge_len):
                local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                
                if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                    continue
                
                start_row, start_col = get_starting_point(S_single, local_row, local_col)
                
                # Case 1: Valid start on bottom edge
                if is_on_bottom_edge(start_row, start_col, D_single.shape):
                    D_chunks[0][j][edge_type][position] = D_single[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[0][j][edge_type][position] = path_length
                
                # Case 2: Start on left edge - need cost from previous chunk
                elif is_on_left_edge(start_row, start_col, D_single.shape):
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        0, j, start_row, start_col, chunks_dict
                    )
                    
                    # Map to previous chunk (0, j-1)
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, 0, j - 1, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[0][j-1][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get cost from previous chunk
                    prev_cost = D_chunks[0][j-1][prev_edge_type][prev_position]
                    
                    if np.isfinite(prev_cost):  # Only if valid path exists
                        # Subtract overlapping cell cost from CURRENT chunk to avoid double counting
                        overlap_cost = C_chunk[start_row, 0] if C_chunk is not None else 0
                        
                        D_chunks[0][j][edge_type][position] = D_single[local_row, local_col] + prev_cost - overlap_cost 
                        # Path length includes previous chunk
                        prev_length = L_chunks[0][j-1][prev_edge_type][prev_position]
                        curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[0][j][edge_type][position] = prev_length + curr_length
    
    # Initialize first column: chunks (i, 0) for i = 1, 2, ...
    for i in range(1, num_chunks_1):
        if (i, 0) not in chunks_dict:
            continue
            
        D_single = chunks_dict[(i, 0)]['D']
        S_single = chunks_dict[(i, 0)]['S']
        C_chunk = chunks_dict[(i, 0)]['C'] if 'C' in chunks_dict[(i, 0)] else None
        
        for edge_type in range(2):
            edge_len = get_edge_length(D_single.shape, edge_type)
            D_chunks[i][0][edge_type] = [np.inf] * edge_len
            L_chunks[i][0][edge_type] = [np.inf] * edge_len
            
            for position in range(edge_len):
                local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                
                if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                    continue
                
                start_row, start_col = get_starting_point(S_single, local_row, local_col)
                
                # Case 1: Valid start on left edge
                if is_on_left_edge(start_row, start_col, D_single.shape):
                    D_chunks[i][0][edge_type][position] = D_single[local_row, local_col]
                    path_length = abs(local_row - start_row) + abs(local_col - start_col)
                    L_chunks[i][0][edge_type][position] = path_length
                
                # Case 2: Start on bottom edge - need cost from previous chunk
                elif is_on_bottom_edge(start_row, start_col, D_single.shape):
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        i, 0, start_row, start_col, chunks_dict
                    )
                    
                    # Map to previous chunk (i-1, 0)
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, i - 1, 0, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[i-1][0][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get cost from previous chunk
                    prev_cost = D_chunks[i-1][0][prev_edge_type][prev_position]
                    
                    if np.isfinite(prev_cost):  # Only if valid path exists
                        # Subtract overlapping cell cost from CURRENT chunk
                        overlap_cost = C_chunk[0, start_col] if C_chunk is not None else 0
                        
                        D_chunks[i][0][edge_type][position] = D_single[local_row, local_col] + prev_cost - overlap_cost
                        
                        # Path length includes previous chunk
                        prev_length = L_chunks[i-1][0][prev_edge_type][prev_position]
                        curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[i][0][edge_type][position] = prev_length + curr_length
    
    return D_chunks, L_chunks


def dp_fill_chunks(chunks_dict, D_chunks, L_chunks, num_chunks_1, num_chunks_2, L):
    """
    Fill in D_chunks and L_chunks for all interior chunks using dynamic programming.
    Uses flexible hop sizes from chunks_dict.
    
    Parameters:
    -----------
    chunks_dict : dict
        Dictionary containing chunk data (including flexible 'hop' per chunk)
    D_chunks, L_chunks : list of list of dicts of lists
        Chunk-level cost and length tensors (partially filled)
    num_chunks_1, num_chunks_2 : int, int
        Number of chunks in each dimension
    L : int
        Standard chunk size (for reference)
    """
    # plot_normalized_global_edge_cost(D_chunks, L_chunks, num_chunks_1, num_chunks_2)
    # Helper function to get actual edge length for a chunk
    def get_edge_length(chunk_shape, edge_type):
        """Returns the actual length of an edge for a given chunk shape."""
        if edge_type == 0:  # top edge
            return chunk_shape[1]  # width
        else:  # right edge (edge_type == 1)
            return chunk_shape[0]  # height
    
    # Process chunks row by row
    for i in range(num_chunks_1):
        for j in range(num_chunks_2):
            # Skip first row and first column (already initialized)
            if i == 0 or j == 0:
                continue
            
            D_single = chunks_dict[(i, j)]['D']
            S_single = chunks_dict[(i, j)]['S']
            C_chunk = chunks_dict[(i, j)]['C'] if 'C' in chunks_dict[(i, j)] else None
            
            for edge_type in range(2):
                edge_len = get_edge_length(D_single.shape, edge_type)
                # Initialize lists for this chunk if not already done
                D_chunks[i][j][edge_type] = [np.inf] * edge_len
                L_chunks[i][j][edge_type] = [np.inf] * edge_len
                
                for position in range(edge_len):
                    
                        
                        # plot_normalized_global_edge_cost(D_chunks, L_chunks, num_chunks_1, num_chunks_2)
                    local_row, local_col = convert_edge_to_local(edge_type, position, D_single.shape)
                    
                    if local_row >= D_single.shape[0] or local_col >= D_single.shape[1]:
                        continue
                    
                    # Get starting point for this path
                    start_row, start_col = get_starting_point(S_single, local_row, local_col)
                    
                    # Determine which previous chunk this path came from
                    if is_on_bottom_edge(start_row, start_col, D_single.shape):
                        # Path came from chunk above (i-1, j)
                        prev_i, prev_j = i - 1, j
                    elif is_on_left_edge(start_row, start_col, D_single.shape):
                        # Path came from chunk to the left (i, j-1)
                        prev_i, prev_j = i, j - 1
                    else:
                        # Path started within this chunk
                        D_chunks[i][j][edge_type][position] = D_single[local_row, local_col]
                        path_length = abs(local_row - start_row) + abs(local_col - start_col)
                        L_chunks[i][j][edge_type][position] = path_length
                        continue
                    
                    # Get global coordinates of starting point
                    global_start_row, global_start_col = convert_local_to_global(
                        i, j, start_row, start_col, chunks_dict
                    )
                    
                    # Map to edge position in previous chunk
                    prev_edge_type, prev_position = global_to_prev_chunk_edge(
                        global_start_row, global_start_col, prev_i, prev_j, chunks_dict, L
                    )
                    
                    # Check if prev_position is within bounds of previous chunk's edge
                    prev_edge_len = len(D_chunks[prev_i][prev_j][prev_edge_type])
                    if prev_position >= prev_edge_len:
                        continue
                    
                    # Get accumulated cost and length from previous chunk
                    prev_cost = D_chunks[prev_i][prev_j][prev_edge_type][prev_position]
                    prev_length = L_chunks[prev_i][prev_j][prev_edge_type][prev_position]
                    
                    # Only proceed if valid path exists in previous chunk
                    if not np.isfinite(prev_cost):
                        continue
                    
                    # Current chunk contribution (subtract first cell to avoid double counting)
                    if C_chunk is not None:
                        first_cell_cost = C_chunk[start_row, start_col]
                    else:
                        first_cell_cost = 0
                    
                    curr_cost_contribution = D_single[local_row, local_col] - first_cell_cost
                    curr_length = abs(local_row - start_row) + abs(local_col - start_col)
                    
                    # Propagate forward
                    
                    D_chunks[i][j][edge_type][position] = prev_cost + curr_cost_contribution
                    if position%4000==0:
                        print(D_chunks[i][j][edge_type][position])
                        print(i,j,edge_type,position)
                    L_chunks[i][j][edge_type][position] = prev_length + curr_length
    
    return D_chunks, L_chunks
def chunked_flexdtw(chunks_dict, L, num_chunks_1, num_chunks_2, buffer_param=0.1):
    """
    Run the complete chunked FlexDTW algorithm with flexible hop sizes.
    
    Parameters:
    -----------
    chunks_dict : dict
        Dictionary with chunk data (including flexible 'hop' per chunk)
    L : int
        Standard chunk size
    num_chunks_1, num_chunks_2 : int, int
        Number of chunks in each dimension
    buffer_param : float
        Buffer parameter
        
    Returns:
    --------
    D_chunks, L_chunks : Filled cost and length tensors
    """
    print("Stage 1: Initialization...")
    D_chunks, L_chunks = initialize_chunks(chunks_dict, num_chunks_1, num_chunks_2, L) 
    
    print("Stage 2: Dynamic Programming...")
    D_chunks, L_chunks = dp_fill_chunks(chunks_dict, D_chunks, L_chunks, 
                                        num_chunks_1, num_chunks_2, L)
     
    print(f"Final chunk edge length: {len(D_chunks[-1][-1][1])}")
    
    return D_chunks, L_chunks

In [15]:
import os
import numpy as np

# Input directory (where the npy files are)
BASE_DIR = "/home/ijain/parflex/sample_recordings_0.2a"

# Output directory (all results go here)
OUT_DIR = "/home/ijain/parflex/sample_recordings_0.2a_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

for root, dirs, files in os.walk(BASE_DIR):
    
    # find .npy files in this folder
    npy_files = [f for f in files if f.endswith(".npy")]
    
    # only process folders with exactly 2 npy files
    if len(npy_files) != 2:
        continue

    # load files
    f1_path = os.path.join(root, npy_files[0])
    f2_path = os.path.join(root, npy_files[1])

    print(f"Processing folder: {root}")
    print(f"  Files: {npy_files[0]}, {npy_files[1]}")

    F1 = np.load(f1_path)
    F2 = np.load(f2_path)

    # create a unique output subfolder mirroring the input structure
    rel_path = os.path.relpath(root, BASE_DIR)
    out_subdir = os.path.join(OUT_DIR, rel_path)
    os.makedirs(out_subdir, exist_ok=True)

    # filenames
    outfile_test = os.path.join(out_subdir, "aligned_result.pkl")
    out_D = os.path.join(out_subdir, "D_chunks.npy")
    out_L = os.path.join(out_subdir, "L_chunks.npy")

    # ---- RUN YOUR PIPELINE ----
    chunks_dict, L, n_chunks_1, n_chunks_2 = align_system(
        "flexdtw", F1, F2, outfile_test, L=4000
    )

    D_chunks, L_chunks = chunked_flexdtw(
        chunks_dict, L, n_chunks_1, n_chunks_2, buffer_param=1
    )

    # ---- SAVE OUTPUT FILES ----
    np.save(out_D, D_chunks)
    np.save(out_L, L_chunks)

    print(f"  Saved outputs in: {out_subdir}")


Processing folder: /home/ijain/parflex/sample_recordings_0.2a/post_20
  Files: Chopin_Op063No3_Magaloff-1978_pid9074g-12.npy, Chopin_Op063No3_Csalog-1996_pid1263b-12.npy
Matrix shape: 6720 × 6239
Chunk (0,0): [0:4000, 0:4000], shape=(4000, 4000), hop=(3999,3999), best_cost=0.0932
Chunk (0,1): [0:4000, 3999:6239], shape=(4000, 2240), hop=(3999,2240), best_cost=0.1461
Chunk (1,0): [3999:6720, 0:4000], shape=(2721, 4000), hop=(2721,3999), best_cost=0.0975
Chunk (1,1): [3999:6720, 3999:6239], shape=(2721, 2240), hop=(2721,2240), best_cost=0.2642
Stage 1: Initialization...
Stage 2: Dynamic Programming...
1.0
1 1 0 0
1.0
1 1 1 0
Final chunk edge length: 2721
  Saved outputs in: /home/ijain/parflex/sample_recordings_0.2a_outputs/post_20
Processing folder: /home/ijain/parflex/sample_recordings_0.2a/subseq_20
  Files: Chopin_Op024No2_Smith-1975_pid9054-15.npy, Chopin_Op024No2_Perlemuter-1992_pid9089-08.npy
Matrix shape: 862 × 862
Chunk (0,0): [0:862, 0:862], shape=(862, 862), hop=(862,862), bes

In [ ]:
D_chunks

NameError: name 'D_chunjs' is not defined